# VASP Gemma E2B QLoRA Training (Planner + Refiner)

This notebook trains two adapters on your paired datasets:
- Planner adapter (`planner_e2b_qlora`)
- Refiner adapter (`refiner_e2b_qlora`)

It assumes these files exist in repo:
- `finetune/configs/train_planner_paired_qlora.yaml`
- `finetune/configs/train_refiner_paired_qlora.yaml`
- `finetune/data/planner_paired_train.jsonl`
- `finetune/data/planner_paired_val.jsonl`
- `finetune/data/refiner_paired_train.jsonl`
- `finetune/data/refiner_paired_val.jsonl`


In [1]:
# 1) Check GPU runtime
!nvidia-smi


Mon Apr 20 22:31:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# 2) Clone repo (edit URL)
REPO_URL = "https://github.com/theextraordinary/VASP.git"
REPO_DIR = "/content/VASP"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already exists, pulling latest...")
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}


Cloning into '/content/VASP'...
remote: Enumerating objects: 449, done.
remote: Counting objects: 100% (449/449), done.
remote: Compressing objects: 100% (346/346), done.
remote: Total 449 (delta 116), reused 425 (delta 95), pack-reused 0 (from 0)
Receiving objects: 100% (449/449), 6.45 MiB | 37.10 MiB/s, done.
Resolving deltas: 100% (116/116), done.
/content/VASP


In [3]:
# 3) Install dependencies
!pip -q install transformers datasets peft accelerate bitsandbytes trl pyyaml huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 51.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 50.1 MB/s eta 0:00:00:00:0100:01


In [4]:
# 4) Hugging Face login (needed for google/gemma-4-e2b-it)
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [5]:
# 5) Verify required files
from pathlib import Path
required = [
    "finetune/configs/train_planner_paired_qlora.yaml",
    "finetune/configs/train_refiner_paired_qlora.yaml",
    "finetune/data/planner_paired_train.jsonl",
    "finetune/data/planner_paired_val.jsonl",
    "finetune/data/refiner_paired_train.jsonl",
    "finetune/data/refiner_paired_val.jsonl",
]
missing = [f for f in required if not Path(f).exists()]
if missing:
    print("Missing files:")
    for m in missing:
        print(" -", m)
    raise SystemExit("Please upload/sync these files before training.")
print("All required files found.")


All required files found.


In [6]:
# 6) Optional sanity-check dataset counts
import json
from pathlib import Path
for f in [
    "finetune/data/planner_paired_train.jsonl",
    "finetune/data/planner_paired_val.jsonl",
    "finetune/data/refiner_paired_train.jsonl",
    "finetune/data/refiner_paired_val.jsonl",
]:
    n = sum(1 for _ in Path(f).open("r", encoding="utf-8"))
    print(f"{f}: {n}")


finetune/data/planner_paired_train.jsonl: 399
finetune/data/planner_paired_val.jsonl: 21
finetune/data/refiner_paired_train.jsonl: 399
finetune/data/refiner_paired_val.jsonl: 21


## Train Planner Adapter


In [7]:
!pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-0m444wj1
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-0m444wj1
  Resolved https://github.com/huggingface/transformers.git to commit ef97a7525dc6e4665db4ba5e1ec40d79e43cc74e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.6.0.dev0-py3-none-any.whl size=11450409 sha256=64135b922c9f4ae69e00943d7d90d255a5907c18cb0f023d2354a230684f50f5
  Stored in directory: /tmp/pip-ephem-wheel-cache-ktg69qjf/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [8]:
# 7) Train planner
!python -m finetune.training.train_qlora --config finetune/configs/train_planner_paired_qlora.yaml


config.json: 4.95kB [00:00, 3.80MB/s]
tokenizer_config.json: 2.10kB [00:00, 12.3MB/s]
tokenizer.json: 100% 32.2M/32.2M [00:00<00:00, 39.5MB/s]
chat_template.jinja: 16.3kB [00:00, 91.6MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 10.2G/10.2G [00:22<00:00, 457MB/s] 
Loading weights: 100% 1951/1951 [00:02<00:00, 835.63it/s] 
generation_config.json: 100% 208/208 [00:00<00:00, 2.25MB/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/VASP/finetune/training/train_qlora.py", line 88, in <module>
    main()
  File "/content/VASP/finetune/training/train_qlora.py", line 29, in main
    model, tokenizer = load_model_and_tokenizer(cfg)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/VASP/finetune/training/model_setup.py", line 52, in load_model_and_tokenizer
    model = get_peft_model(model, lora_config)
            ^^^^

## Train Refiner Adapter


In [ ]:
# 8) Train refiner
!python -m finetune.training.train_qlora --config finetune/configs/train_refiner_paired_qlora.yaml


In [ ]:
# 9) (Optional) Run evaluation if you have eval config wired to new outputs
# !python -m finetune.evaluation.run_eval --config finetune/configs/eval.yaml


In [ ]:
# 10) Zip trained adapters
!zip -r planner_e2b_qlora.zip finetune/output/planner_e2b_qlora
!zip -r refiner_e2b_qlora.zip finetune/output/refiner_e2b_qlora


In [ ]:
# 11) Download to local machine
from google.colab import files
files.download("planner_e2b_qlora.zip")
files.download("refiner_e2b_qlora.zip")


In [ ]:
# 12) (Optional) Save artifacts to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp planner_e2b_qlora.zip /content/drive/MyDrive/
# !cp refiner_e2b_qlora.zip /content/drive/MyDrive/
